In [16]:
!pip install rank_bm25
!pip install scikit-learn


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import import_ipynb
from preprocessing import *

In [2]:
import pandas as pd
from collections import Counter
from tqdm import tqdm
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [15]:
%store -r en_data

### Частотности

In [8]:
def get_freq_data():

    matrix_data = {'texts':[], 'words':[]}
    
    for i, row in tqdm(en_data.iterrows()):
      clean_text = row['cleaned_text']
      original_text = row['target']
      frequencies = dict(Counter(clean_text.split()))
    
      matrix_data['texts'].append(original_text) # хотим находить изначальный, а не нормализированный текст
    
      matrix_data['words'].append(frequencies)
    
    freq_data = pd.DataFrame(matrix_data)
    
    freq_data = pd.json_normalize(freq_data['words']).set_index(freq_data['texts']).fillna(0.0)

    return freq_data

### bm25

In [12]:
clean_text = en_data['cleaned_text']
clean_split = [text.split() for text in clean_text]
bm25 = BM25Okapi(clean_split)

### Функция поиска

In [17]:
def search(query:str, index_type:str='bm25', data=en_data, top_k:int=5):
  
  '''
  Принимает запрос, вид обратного индекса, data изначальных текстов для поиска и число выданных текстов по убыванию ранжирования
  Аргументы:
  query:str -- запрос
  data:df -- тексты для поиска, обязательны столбцы cleaned_texts и target с изначальными текстами
  index_type:str -- вид индекса, варианты bm25 или freq (частоты)
  top_k:int -- число топ-текстов по актуальности запросу

  Возвращает датафрейм с изначальными текстами, отранжированными по близости к запросу и столбцами-значениями близости
  '''

  clean_query = cleaner(query) 
  if type(clean_query) != str:
      print(clean_query)
  freq_data = get_freq_data()

  if index_type == 'bm25':
      
    query = clean_query.split()
    bm_scores = bm25.get_scores(query)

    data['bm_sim'] = bm_scores

    ranked_data = data.sort_values(by='bm_sim', ascending=False) 

    return ranked_data.head(top_k).reset_index(drop=True)[['target', 'bm_sim']]

  elif index_type == 'freq':

    clean_queries = clean_query.split()
    clean_queries = [w for w in clean_queries if w in freq_data.columns] # убираем слова, отсутствующие в изначальном корпусе

    if len(clean_queries) == 0:
      return 'Ни одно из слов запроса не присутствует в изначальном корпусе'

    elif len(clean_queries) == 1:
      return freq_data[query].sort_values(ascending=False).dropna().reset_index().head(top_k)

    else:
      sample_data = freq_data[clean_queries]

      sample_data['sum'] = sample_data.sum(axis=1)
      sample_data['both'] = sample_data[clean_queries].notna().all(axis=1).astype(int)

      sort_data = sample_data.sort_values(by='sum',
                      key=lambda x: sample_data.apply(lambda row: (row['both'], row['sum'], row[sample_data.columns[0]]),
                      axis=1),
                      ascending=False
                      ).drop('both', axis=1)

      return sort_data.reset_index().head(top_k)

  else:
    raise ValueError(f'Выбран неверный вариант индекса -- {index_type}, доступные варианты: bm25, freq')